# Ingeniería de features con SQL analítico avanzado

## 1. Objetivo del análisis

El objetivo de este notebook es construir variables de comportamiento a partir del historial de citas médicas de los pacientes, utilizando SQL analítico y DuckDB.

La pregunta que orienta el análisis es:

**¿Qué patrones históricos de asistencia pueden ayudar a caracterizar la probabilidad de que un paciente falte a una cita médica posterior?**

A partir del dataset *Medical Appointment No Shows*, se calcularán variables relacionadas con la recencia, la frecuencia de visitas, la tasa histórica de inasistencia y la tendencia de actividad.

## 2. Metodología

Las transformaciones se realizarán mediante consultas SQL, con especial atención al uso de funciones de ventana para analizar el historial de cada paciente.

Cada cita se evaluará utilizando exclusivamente información de citas anteriores. El resultado de la cita actual (`No-show`) se conservará como variable objetivo, pero no se utilizará para calcular sus propias features, evitando así la fuga de información (*data leakage*).

El análisis se ejecutará en DuckDB desde Python, utilizando Jupyter Notebook como entorno de trabajo.


In [1]:
import pandas as pd
from pathlib import Path

# Comprobar la carpeta de trabajo
print("Carpeta actual:", Path.cwd())

# Comprobar si DuckDB está disponible
try:
    import duckdb
    print("DuckDB instalado. Versión:", duckdb.__version__)
except ImportError:
    print("DuckDB no está instalado.")

Carpeta actual: C:\Users\luisi\Desktop\Bioinformatica\Data Science\Data science II\patient-attendance-feature-engineering\notebooks
DuckDB instalado. Versión: 1.5.6


In [4]:
proyecto = Path.cwd().parent
ruta_csv = proyecto / "data" / "KaggleV2-May-2016.csv"

print("Ruta del CSV:", ruta_csv)
print("¿Existe el archivo?", ruta_csv.exists())

Ruta del CSV: C:\Users\luisi\Desktop\Bioinformatica\Data Science\Data science II\patient-attendance-feature-engineering\data\KaggleV2-May-2016.csv
¿Existe el archivo? True


## 3. Configuración y acceso a los datos

Se establece una conexión con DuckDB y se crea una vista SQL sobre el archivo CSV original. Esto permite ejecutar consultas analíticas directamente sobre los datos, sin generar una copia completa del dataset ni crear una base de datos persistente.


In [7]:
# Crear una conexión DuckDB en memoria
con = duckdb.connect()

# Registrar el CSV como una vista consultable desde SQL
ruta_sql = ruta_csv.as_posix().replace("'", "''")

con.execute(f"""
    CREATE VIEW appointments AS
    SELECT *
    FROM read_csv_auto('{ruta_sql}')
""")

print("Vista 'appointments' creada correctamente.")

Vista 'appointments' creada correctamente.


In [8]:
consulta = """
SELECT
    PatientId,
    AppointmentID,
    ScheduledDay,
    AppointmentDay,
    "No-show"
FROM appointments
LIMIT 5;
"""

display(con.execute(consulta).df())

,PatientId,AppointmentID,ScheduledDay,AppointmentDay,No-show
0,2.987250e+13,5642903,2016-04-29 19:38:08+01:00,2016-04-29 01:00:00+01:00,False
1,5.589978e+14,5642503,2016-04-29 17:08:27+01:00,2016-04-29 01:00:00+01:00,False
2,4.262962e+12,5642549,2016-04-29 17:19:04+01:00,2016-04-29 01:00:00+01:00,False
3,8.679512e+11,5642828,2016-04-29 18:29:31+01:00,2016-04-29 01:00:00+01:00,False
4,8.841186e+12,5642494,2016-04-29 17:07:23+01:00,2016-04-29 01:00:00+01:00,False


## 4. Construcción de features mediante SQL analítico

Se construirán variables históricas por paciente utilizando funciones de ventana. Cada fila representa una cita médica y las variables predictivas se calcularán exclusivamente a partir de citas anteriores a la cita evaluada.

Se utilizarán las siguientes métricas:

* **Recency:** días desde la cita anterior.
* **Frequency:** cantidad de citas anteriores.
* **Tasa histórica de inasistencia:** proporción de citas anteriores con resultado `No-show = Yes`.
* **Trend:** diferencia entre la actividad registrada en los 30 días previos y la de los 30 días anteriores a ese período.

La tasa histórica de inasistencia se utiliza como adaptación de una métrica de comportamiento, ya que el dataset no contiene datos monetarios para calcular Monetary de manera válida.


### 4.1. Ejecución de la consulta analítica

La consulta SQL se almacena en un archivo independiente para separar la lógica de transformación del código Python. DuckDB ejecuta las instrucciones sobre la vista `appointments` y devuelve una tabla con la variable objetivo y las features históricas generadas.


In [10]:
from pathlib import Path

ruta_sql = proyecto / "sql" / "feature_engineering.sql"

consulta_features = ruta_sql.read_text(encoding="utf-8")

df_features = con.execute(consulta_features).df()

print("Consulta ejecutada correctamente.")
print("Dimensiones del resultado:", df_features.shape)

display(df_features.head())

Consulta ejecutada correctamente.
Dimensiones del resultado: (110522, 11)


,patient_id,appointment_id,scheduled_date,appointment_date,no_show,recency_days,frequency,historical_no_show_rate,visits_last_30_days,visits_previous_30_days,visit_trend
0,39218,5751990,2016-05-31,2016-06-03,0,<NA>,0.0,NaN,0.0,0.0,0.0
1,43742,5760144,2016-06-01,2016-06-01,0,<NA>,0.0,NaN,0.0,0.0,0.0
2,93780,5712759,2016-05-18,2016-05-18,0,<NA>,0.0,NaN,0.0,0.0,0.0
3,141724,5637648,2016-04-29,2016-05-02,0,<NA>,0.0,NaN,0.0,0.0,0.0
4,537615,5637728,2016-04-29,2016-05-06,0,<NA>,0.0,NaN,0.0,0.0,0.0


### 4.2. Primer resultado de la ingeniería de features

La consulta genera una tabla con una fila por cita válida, la variable objetivo `no_show` y las variables históricas calculadas mediante SQL analítico.

Para los pacientes sin historial previo, la frecuencia se establece en cero, mientras que la recencia y la tasa histórica de inasistencia permanecen nulas porque no existen observaciones anteriores con las que calcularlas.

La validez de estas variables se comprobará mediante controles de integridad y consistencia temporal.


In [11]:
# Cantidad de filas y citas únicas
print("Filas generadas:", len(df_features))
print("Citas únicas:", df_features["appointment_id"].nunique())
print(
    "IDs de cita duplicados:",
    df_features["appointment_id"].duplicated().sum()
)

# Valores nulos de las variables generadas
print("\nValores nulos por columna:")
display(df_features.isna().sum())

# Distribución de la frecuencia histórica
print("\nDistribución de frecuencia histórica:")
display(df_features["frequency"].describe())

# Cantidad de citas con y sin historial
print("\nCitas con historial previo:")
print((df_features["frequency"] > 0).sum())

print("\nCitas sin historial previo:")
print((df_features["frequency"] == 0).sum())

Filas generadas: 110522
Citas únicas: 110522
IDs de cita duplicados: 0

Valores nulos por columna:


patient_id                     0
appointment_id                 0
scheduled_date                 0
appointment_date               0
no_show                        0
recency_days               66289
frequency                      0
historical_no_show_rate    66289
visits_last_30_days            0
visits_previous_30_days        0
visit_trend                    0
dtype: int64


Distribución de frecuencia histórica:


count    110522.000000
mean          1.176445
std           3.797305
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          83.000000
Name: frequency, dtype: float64


Citas con historial previo:
44233

Citas sin historial previo:
66289


In [12]:
columnas_revision = [
    "patient_id",
    "appointment_id",
    "appointment_date",
    "no_show",
    "recency_days",
    "frequency",
    "historical_no_show_rate",
    "visits_last_30_days",
    "visits_previous_30_days",
    "visit_trend"
]

display(
    df_features.loc[
        df_features["frequency"] > 0,
        columnas_revision
    ].head(10)
)

,patient_id,appointment_id,appointment_date,no_show,recency_days,frequency,historical_no_show_rate,visits_last_30_days,visits_previous_30_days,visit_trend
8,22638656,5715081,2016-06-08,0,36,1.0,0.0,0.0,1.0,-1.0
10,52168938,5607220,2016-05-17,0,1,1.0,0.0,1.0,0.0,1.0
15,64851211,5697532,2016-05-17,0,4,1.0,1.0,1.0,0.0,1.0
18,79228495,5743266,2016-06-08,0,9,1.0,0.0,1.0,0.0,1.0
21,87996454,5786272,2016-06-08,0,26,1.0,1.0,1.0,0.0,1.0
25,122451254,5576521,2016-05-09,1,7,1.0,1.0,1.0,0.0,1.0
28,142133299,5767863,2016-06-07,0,22,1.0,1.0,1.0,0.0,1.0
30,156223164,5729757,2016-05-24,0,11,1.0,0.0,1.0,0.0,1.0
35,188964456,5724671,2016-05-20,0,2,1.0,0.0,1.0,0.0,1.0
36,188964456,5729804,2016-05-24,0,4,2.0,0.0,2.0,0.0,2.0


In [13]:
# 1. Comprobar que las features de recencia y tasa histórica
# sean nulas solamente cuando no existe historial previo.

sin_historial = df_features["frequency"] == 0
con_historial = df_features["frequency"] > 0

print(
    "Sin historial y recencia nula:",
    (
        sin_historial
        & df_features["recency_days"].isna()
    ).sum()
)

print(
    "Con historial y recencia disponible:",
    (
        con_historial
        & df_features["recency_days"].notna()
    ).sum()
)

print(
    "Con historial y tasa histórica disponible:",
    (
        con_historial
        & df_features["historical_no_show_rate"].notna()
    ).sum()
)

# 2. Verificar que la tasa histórica esté entre 0 y 1.
tasas_validas = df_features["historical_no_show_rate"].dropna()

print("\nTasa histórica dentro del rango [0, 1]:")
print(tasas_validas.between(0, 1).all())

# 3. Verificar que las features de actividad no sean negativas.
columnas_actividad = [
    "frequency",
    "visits_last_30_days",
    "visits_previous_30_days"
]

print("\nTodas las métricas de actividad son no negativas:")
print((df_features[columnas_actividad] >= 0).all().all())

# 4. Revisar la cantidad de registros por variable objetivo.
print("\nDistribución de la variable objetivo:")
display(df_features["no_show"].value_counts().sort_index())

Sin historial y recencia nula: 66289
Con historial y recencia disponible: 44233
Con historial y tasa histórica disponible: 44233

Tasa histórica dentro del rango [0, 1]:
True

Todas las métricas de actividad son no negativas:
True

Distribución de la variable objetivo:


no_show
0    88208
1    22314
Name: count, dtype: int64

## 5. Validación de las features

Los controles realizados confirman que el resultado contiene 110.522 citas únicas, sin identificadores duplicados. Se excluyeron cinco registros con inconsistencias temporales detectadas durante la exploración.

Las 44.233 citas con historial previo cuentan con valores disponibles para `recency_days` y `historical_no_show_rate`. Para las 66.289 citas sin historial, ambas variables permanecen nulas porque no existen observaciones anteriores para calcularlas. Las métricas de actividad, en cambio, utilizan cero cuando no se registran visitas previas en el período correspondiente.

La tasa histórica de inasistencia se encuentra dentro del intervalo de 0 a 1 y las métricas de actividad no presentan valores negativos.

La variable objetivo conserva 88.208 asistencias y 22.314 inasistencias entre las citas válidas. Las features históricas se mantienen separadas del resultado de la cita evaluada para evitar utilizar ese resultado en su propio cálculo.


## 6. Conclusiones

Se construyeron variables históricas mediante SQL analítico y DuckDB para caracterizar el comportamiento de los pacientes respecto de sus citas médicas.

Las features generadas son:

* `recency_days`: días desde la cita anterior.
* `frequency`: cantidad de citas anteriores.
* `historical_no_show_rate`: proporción histórica de inasistencias.
* `visits_last_30_days`: cantidad de citas durante los 30 días previos.
* `visits_previous_30_days`: cantidad de citas en los 30 días anteriores a ese período.
* `visit_trend`: diferencia entre la actividad de ambos períodos.

La consulta utiliza funciones de ventana como `LAG()` y `SUM() OVER()`, además de marcos temporales para calcular la actividad histórica. Se verificó que las citas sean únicas y que las métricas generadas respeten los rangos esperados.

La tasa histórica de inasistencia y la recencia permanecen nulas cuando no existe historial previo suficiente para calcularlas. Las métricas de actividad utilizan cero cuando no se registran visitas en los períodos analizados.

Estas variables permiten preparar un conjunto de datos para estudiar la relación entre el historial de citas y las inasistencias. Sin embargo, su utilidad predictiva deberá evaluarse posteriormente mediante un análisis o modelo específico; la ingeniería de features por sí sola no demuestra capacidad predictiva.


In [14]:
import sys
import pandas as pd
import duckdb
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("duckdb:", duckdb.__version__)

for paquete in ["jupyterlab", "ipykernel"]:
    try:
        print(f"{paquete}: {version(paquete)}")
    except PackageNotFoundError:
        print(f"{paquete}: no instalado en este entorno")

Python: 3.11.16
pandas: 3.0.6
duckdb: 1.5.6
jupyterlab: 4.6.4
ipykernel: 7.3.0
